# OGDT Member Behavior: Monthly Gym Attendance

Real historical fitness data from Stefano DellaVigna and Ulrike Malmendier,
[Paying Not to Go to the Gym](https://eml.berkeley.edu/~sdellavi/data/gymdata.shtml).
This notebook studies recorded attendance trajectories, with **unknown months kept distinct from recorded zeros**.
It is a descriptive research component of Open Gym Digital Twin, not a validated engagement or churn model.

The original study already examines attendance and contracts. Our contribution is a reusable, audited
12-metric monthly interface, gap-aware windows and explicit integration boundaries. We do not claim a new
scientific discovery or reproduce the paper's 7,752-member sample.

The source is downloaded directly from the author's site into temporary storage and checked by SHA-256.
No source files or member-level exports are bundled or written to notebook outputs. Public access to the
source is not an explicit redistribution license. Code licensing does not grant rights to the underlying data.
Enable Internet on Kaggle. Alternatively set `OGDT_BERKELEY_ARCHIVE` to a lawfully obtained local ZIP.


In [ ]:
from pathlib import Path
import os, json, tempfile, urllib.request, platform, importlib.metadata
from IPython.display import display
print(json.dumps({"python": platform.python_version(), **{p: importlib.metadata.version(p) for p in ["numpy", "pandas", "matplotlib"]}}, indent=2))


In [ ]:
"""Pinned real-fitness monthly source. Never interpret calendar padding as capture."""
from pathlib import Path
from datetime import datetime, timedelta
import hashlib
import io
import zipfile
import numpy as np
import pandas as pd

DATASET_ID = "berkeley_gym_aer_2006"
ARCHIVE_SHA256 = "f2abbaa0312b697c56892621782073a8dd4b0fbf7cdbb8b9746a28c6398772f9"
TABLE_SHA256 = "e56ffe6177eb5e1508555d07a756a58f9e74d051eed12601ab5a34919949289c"
POLICY_ID = "berkeley_regular_month_v1"
KEY = ["dataset_id", "member_id", "period_end"]
PAPER_START = pd.Timestamp("1997-04-01")
PAPER_LAST_MONTH = {11: pd.Timestamp("2000-07-01"), 13: pd.Timestamp("2001-02-01"), 16: pd.Timestamp("2001-02-01")}
SOURCE_COLUMNS = {"id", "time", "attend", "contract", "imprice", "trprice", "code", "init", "problem", "club", "birthday", "female", "age", "month", "begatt", "endatt", "endcont", "endcontp", "overtime", "dlocker", "duplicate"}


def matlab_dates(values):
    numeric = pd.to_numeric(values, errors="raise")
    if numeric.isna().any() or not np.isfinite(numeric).all() or (numeric != np.floor(numeric)).any():
        raise ValueError("Expected finite integral MATLAB daily dates")
    mapping = {v: pd.Timestamp(datetime.fromordinal(int(v))-timedelta(days=366)) for v in numeric.unique()}
    return numeric.map(mapping)


def validate_source(frame):
    required = ["id", "time", "attend", "contract", "club", "duplicate", "problem", "overtime", "begatt", "endatt"]
    if not set(required).issubset(frame) or frame[required].isna().any().any():
        raise ValueError("Missing source fields or required values")
    if frame.duplicated(["id", "time"]).any():
        raise ValueError("Source member/month keys must be unique")
    for col in ["id", "attend", "contract", "club", "duplicate", "problem", "overtime"]:
        v = frame[col]
        if not np.isfinite(v).all() or (v != np.floor(v)).any() or (v < 0).any():
            raise ValueError(f"Invalid nonnegative integral field: {col}")
    if (frame.id <= 0).any() or not frame.club.isin(PAPER_LAST_MONTH).all():
        raise ValueError("Unknown member/club identity")
    if not frame.duplicate.isin([0, 1]).all() or not frame.overtime.isin([0, 1]).all():
        raise ValueError("Unknown source binary flag")
    if not frame.contract.isin([0, 1, 2, 7, 11, 14, 15, 19, 20, 21, 24, 25, 27, 70, 77, 111]).all():
        raise ValueError("Review undocumented source contract code before use")
    dates = matlab_dates(frame.time)
    if not dates.dt.day.eq(1).all():
        raise ValueError("Expected source calendar months, not visit timestamps")
    return dates


def load_berkeley(path):
    content = Path(path).read_bytes()
    if hashlib.sha256(content).hexdigest() != ARCHIVE_SHA256:
        raise ValueError("Source archive hash changed; reassess before processing")
    with zipfile.ZipFile(io.BytesIO(content)) as archive:
        table = archive.read("GymAERSourceData.dta")
        if hashlib.sha256(table).hexdigest() != TABLE_SHA256:
            raise ValueError("Source table hash mismatch")
        with pd.read_stata(io.BytesIO(table), iterator=True, convert_categoricals=False) as reader:
            labels = reader.variable_labels()
            frame = reader.read()
    if set(frame) != SOURCE_COLUMNS:
        raise ValueError("Unexpected source schema")
    validate_source(frame)
    return frame, labels


def monthly_observations(source):
    month = validate_source(source)
    first = matlab_dates(source.begatt)
    last = matlab_dates(source.endatt)
    in_span = month.ge(first) & month.le(last) & month.ge(PAPER_START) & month.le(source.club.map(PAPER_LAST_MONTH))
    review_ok = source.duplicate.eq(0) & source.problem.eq(0) & source.overtime.eq(0)
    regular = source.contract.isin([1, 2])
    eligible = in_span & review_ok & regular
    out = pd.DataFrame({"dataset_id": DATASET_ID,
                        "member_id": source.id.astype("int64").astype(str),
                        "period_end": month.dt.to_period("M").dt.to_timestamp("M"),
                        "club_id": source.club.astype("int64"),
                        "source_contract_code": source.contract.astype("int64"),
                        "excluded_outside_documented_span": ~in_span,
                        "excluded_author_duplicate": source.duplicate.ne(0),
                        "excluded_author_problem": source.problem.ne(0),
                        "excluded_overtime": source.overtime.ne(0),
                        "excluded_nonregular_contract": ~regular,
                        "attendance_eligible": eligible,
                        "recorded_attendance_month": source.attend.where(eligible).astype(float)})
    out["eligibility_status"] = np.where(eligible, "regular_month_in_documented_span", "excluded_see_reason_flags")
    # Reason flags are audit metadata, never behavioral predictors or churn labels.
    return out.sort_values(["member_id", "period_end"], key=lambda s: s.astype(int) if s.name == "member_id" else s).reset_index(drop=True)


def assert_same_population(left, right, left_table_sha256, right_table_sha256):
    """Reject ID-only joins and mismatched populations, even if values coincide."""
    for frame in [left, right]:
        if not set(KEY).issubset(frame) or frame[KEY].isna().any().any() or frame.duplicated(KEY).any():
            raise ValueError("Unique complete namespaced member/month keys are required")
        if not frame.dataset_id.eq(DATASET_ID).all():
            raise ValueError("Unrelated dataset/population: member join prohibited")
    if left_table_sha256 != TABLE_SHA256 or right_table_sha256 != TABLE_SHA256:
        raise ValueError("Source table identity must match the inspected Berkeley table")
    # Matching source identity permits different eligibility subsets. Join only on KEY.
    return left[KEY].merge(right[KEY], on=KEY, how="inner", validate="one_to_one")

"""Calendar-month behavior; unavailable months break histories instead of becoming zero."""
import numpy as np
import pandas as pd

FEATURE_COLUMNS = ["recorded_attendance_month", "attendance_previous_month", "attendance_change_vs_previous_month",
                   "attendance_pct_change_vs_previous_month", "attendance_sum_3_calendar_months",
                   "attendance_mean_3_calendar_months", "attendance_std_3_calendar_months",
                   "positive_attendance_months_3", "attendance_sum_6_calendar_months",
                   "attendance_mean_6_calendar_months", "consecutive_eligible_months",
                   "months_since_recorded_positive_month"]
OUTPUT_COLUMNS = KEY + ["club_id", "as_of_exclusive", "synthetic", "selection_policy_id", "availability_basis", "coverage_basis"] + FEATURE_COLUMNS


def monthly_metrics(observations):
    needed = set(KEY + ["club_id", "attendance_eligible", "recorded_attendance_month"])
    if not needed.issubset(observations) or observations[list(needed)].drop(columns="recorded_attendance_month").isna().any().any():
        raise ValueError("Complete monthly identity and eligibility are required")
    if not observations.dataset_id.eq(DATASET_ID).all() or observations.duplicated(KEY).any():
        raise ValueError("Use one namespaced Berkeley row per member/month")
    out = observations[list(needed)].copy().sort_values(["member_id", "period_end"]).reset_index(drop=True)
    out["period_end"] = pd.to_datetime(out.period_end)
    if not out.period_end.eq(out.period_end.dt.to_period("M").dt.to_timestamp("M")).all():
        raise ValueError("period_end must be a calendar month-end date")
    if not out.attendance_eligible.isin([True, False]).all():
        raise ValueError("Eligibility must be explicit")
    eligible = out.attendance_eligible
    if out.loc[eligible, "recorded_attendance_month"].isna().any() or (out.loc[eligible, "recorded_attendance_month"] < 0).any():
        raise ValueError("Eligible months require valid recorded attendance")
    counts = out.loc[eligible, "recorded_attendance_month"]
    if not np.isfinite(counts).all() or not counts.eq(np.floor(counts)).all():
        raise ValueError("Monthly attendance must be finite nonnegative integer counts")
    out.loc[~eligible, "recorded_attendance_month"] = np.nan
    ordinal = out.period_end.dt.to_period("M").astype("int64")
    previous_period = ordinal.groupby(out.member_id).shift()
    previous_eligible = eligible.groupby(out.member_id).shift(fill_value=False)
    continuous = ordinal.sub(previous_period).eq(1) & eligible & previous_eligible
    # A missing calendar row and an explicit ineligible month both break the run.
    breaks = (~continuous).astype(int).groupby(out.member_id).cumsum()
    out["attendance_previous_month"] = out.recorded_attendance_month.groupby(out.member_id).shift().where(continuous)
    # Avoid rolling through hundreds of thousands of ineligible calendar-padding groups.
    out = out.loc[eligible].copy()
    if out.empty:
        return pd.DataFrame(columns=OUTPUT_COLUMNS)
    ordinal = ordinal.loc[out.index]
    groups = [out.member_id, breaks.loc[out.index]]
    out["consecutive_eligible_months"] = out.groupby(groups, sort=False).cumcount() + 1
    out["attendance_change_vs_previous_month"] = out.recorded_attendance_month - out.attendance_previous_month
    out["attendance_pct_change_vs_previous_month"] = out.recorded_attendance_month / out.attendance_previous_month.replace(0, np.nan) - 1
    for length in [3, 6]:
        rolling = out.groupby(groups, sort=False).recorded_attendance_month.rolling(length, min_periods=length)
        out[f"attendance_sum_{length}_calendar_months"] = rolling.sum().reset_index(level=[0, 1], drop=True).reindex(out.index)
        out[f"attendance_mean_{length}_calendar_months"] = out[f"attendance_sum_{length}_calendar_months"] / length
        if length == 3:
            out["attendance_std_3_calendar_months"] = rolling.std(ddof=0).reset_index(level=[0, 1], drop=True).reindex(out.index)
            positive = out.recorded_attendance_month.gt(0)
            out["positive_attendance_months_3"] = positive.groupby(groups, sort=False).rolling(3, min_periods=3).sum().reset_index(level=[0, 1], drop=True).reindex(out.index)
    last_positive_period = ordinal.where(out.recorded_attendance_month.gt(0)).groupby(groups, sort=False).ffill()
    out["months_since_recorded_positive_month"] = ordinal.sub(last_positive_period)
    out["as_of_exclusive"] = out.period_end + pd.Timedelta(days=1)
    out["synthetic"] = False
    out["selection_policy_id"] = POLICY_ID
    out["availability_basis"] = "assumed_month_close_no_ingestion_clock"
    out["coverage_basis"] = "author_flags_and_documented_month_span"
    # Diagnostic exclusion flags, contract endpoints, prices and demographics are absent.
    return out[OUTPUT_COLUMNS].reset_index(drop=True)

## Eligibility before metrics

The raw table contains calendar padding, nonregular contracts and excluded months.
Select contract codes 1/2, accepted duplicate/problem/overtime flags, and the intersection of
source attendance bounds and the published club spans. These retrospective flags define selection;
they are not predictors available in a live system. The clock is an assumed month-close proxy.
Missing or excluded months break histories. Three/six-month windows require every month to be eligible.


In [ ]:
with tempfile.TemporaryDirectory(prefix="ogdt-source-") as temporary:
    supplied = os.environ.get("OGDT_BERKELEY_ARCHIVE")
    if supplied:
        archive_path = Path(supplied)
    else:
        archive_path = Path(temporary) / "source.zip"
        request = urllib.request.Request("https://eml.berkeley.edu/~sdellavi/data/data.zip", headers={"User-Agent": "OGDT-research-notebook/1.0"})
        with urllib.request.urlopen(request, timeout=90) as response:
            content = response.read(10_000_001)
        if len(content) > 10_000_000:
            raise ValueError("Unexpected source archive size")
        archive_path.write_bytes(content)
    source, labels = load_berkeley(archive_path)
audit = monthly_observations(source)
features = monthly_metrics(audit)
assert len(source) == 491506 and source.id.nunique() == 10241
assert len(features) == 84251 and features.member_id.nunique() == 8073
assert int(features.recorded_attendance_month.sum()) == 397354
assert audit.loc[~audit.attendance_eligible, "recorded_attendance_month"].isna().all()
assert not features.synthetic.any()
summary = {"source_rows": len(source), "source_members": int(source.id.nunique()),
           "eligible_rows": len(features), "eligible_members": int(features.member_id.nunique()),
           "recorded_events": int(features.recorded_attendance_month.sum()),
           "mean_attendance": float(features.recorded_attendance_month.mean()),
           "zero_fraction": float(features.recorded_attendance_month.eq(0).mean()),
           "adjacent_comparisons": int(features.attendance_previous_month.notna().sum()),
           "complete_3_month_windows": int(features.attendance_sum_3_calendar_months.notna().sum()),
           "complete_6_month_windows": int(features.attendance_sum_6_calendar_months.notna().sum()),
           "source_sha256": ARCHIVE_SHA256, "selection_policy": POLICY_ID,
           "synthetic": False, "operational_use_allowed": False,
           "member_level_outputs_written": False}
assert summary["adjacent_comparisons"] == 73352
assert summary["complete_3_month_windows"] == 64347
assert summary["complete_6_month_windows"] == 43690
print(json.dumps(summary, indent=2))
support = features[FEATURE_COLUMNS].agg(["count", "mean", "median"]).T
support.index.name = "metric"
display(support)
Path("ogdt_aggregate_summary.json").write_text(json.dumps(summary, indent=2) + "\n", encoding="utf-8")


In [ ]:
import matplotlib.pyplot as plt
monthly = features.groupby(["period_end", "club_id"]).agg(
    eligible_rows=("member_id", "size"), mean_attendance=("recorded_attendance_month", "mean")).reset_index()
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for club, group in monthly.groupby("club_id"):
    axes[0, 0].plot(group.period_end, group.eligible_rows, label=f"Club {club}")
    axes[0, 1].plot(group.period_end, group.mean_attendance, label=f"Club {club}")
axes[0, 0].set_title("Eligible monthly observations — changing population")
axes[0, 1].set_title("Recorded attendance / eligible month")
axes[0, 0].legend()
axes[1, 0].hist(features.recorded_attendance_month, bins=range(0, 51), color="#235789")
axes[1, 0].set_title("Monthly counts (display range 0–50)")
axes[1, 1].barh(["Current month", "Adjacent month", "Complete 3 months", "Complete 6 months"],
               [84251, 73352, 64347, 43690], color="#16817a")
axes[1, 1].set_title("Metric support: non-null member-month rows")
fig.suptitle("OGDT · Real historical Berkeley data · regular-month descriptive subset")
fig.tight_layout()
fig.savefig("ogdt_monthly_aggregates.png", dpi=140)
plt.show()


## Connection to the gym digital twin

This is the historical monthly **observation layer**: counts, changes, rolling volume, variability and
history support. It cannot reconstruct daily visits, peak hours, physical occupancy or true membership tenure.
Monthly recency is not days since a visit. Zero variability may describe repeated zeros, not a strong habit.

The separate OGDT Retention lab uses the same pinned source. Local integration verified all 81,927
of its eligible member-month keys/current counts. Our 2,324 additional rows reflect its family-as-of
exclusion. Retention also allows prior prorated months in some histories; this notebook requires regular
months throughout. These metrics must not silently replace its trained model's inputs. This standalone
notebook does not read Retention labels, scores or local reports, and does not independently repeat that join.

**Limits:** historical three-club population; selection and capture assumptions; no verified ingestion clock,
uncertainty interval, validated engagement states, prediction, intervention effect or current-gym validation.
The mean is weighted by eligible member-month observations, not equally by people. A recorded zero only
applies to an eligible month. Source dates and author flags condition retrospective eligibility.

**Attribution:** Stefano DellaVigna and Ulrike Malmendier (2006), *Paying Not to Go to the Gym*,
American Economic Review, 96(3), 694–719. [Author data page](https://eml.berkeley.edu/~sdellavi/data/gymdata.shtml).
OGDT code and methodology: OGDT contributors. Underlying data retain their own rights; no data license inferred.
